In [ ]:
# =========================================================
# 1. IMPORTS
# =========================================================
print("Loading imports...")
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') # Suppresses Pandas/Sklearn deprecation warnings for cleaner output

from sklearn.linear_model import LinearRegression
from MultiPiecwiseRegressor import *

# =========================================================
# 2. LOAD DATA
# =========================================================
print("Loading data...")
# parse_dates ensures date columns are loaded as datetime objects, saving a conversion step later
train = pd.read_csv('store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('store-sales-time-series-forecasting/test.csv', parse_dates=['date'])

# =========================================================
# 3. DATA PREPARATION & FEATURE ENGINEERING
# =========================================================
print("Engineering features...")

# The competition metric is Root Mean Squared Logarithmic Error (RMSLE).
# By applying log1p (log(1 + x)) to the target now, we can just use standard 
# RMSE as our loss function in the model.
train['log1p_sales'] = np.log1p(train['sales'])

# Adding an explicit intercept column for our custom LinearRegression models
# since we will set fit_intercept=False later.
train['intercept'] = 1
test['intercept'] = 1

# Placeholders for the test set to match training structure
test['sales'] = 0.0
test['log1p_sales'] = 0.0

# =========================================================
# 4. SETUP ROUTING & ESTIMATORS
# =========================================================
print("Setting up routing logic and base estimators...")

def store_family_router(X):
    """
    Creates routing keys in the format "{store_nbr}_{family}".
    Assumes X is a pandas DataFrame containing these columns.
    """
    # Cast store_nbr to int (in case it was read as float) then to string
    store_str = X['store_nbr'].astype(int).astype(str)
    
    # Cast family to string
    family_str = X['family'].astype(str)
    
    # Concatenate them with an underscore
    routes = store_str + "_" + family_str
    
    # Return as a numpy array so the custom estimator's masking logic works perfectly
    return routes.to_numpy()

# Use the routing function to get an array of all keys in the training data
all_training_keys = store_family_router(train)

# Extract just the unique keys
unique_keys = np.unique(all_training_keys)

# Create the dictionary dynamically with fit_intercept=False 
# because we explicitly provided an 'intercept' column above.
my_estimators = {key: LinearRegression(fit_intercept=False) for key in unique_keys}

# =========================================================
# 5. FULL TRAINING
# =========================================================
print("\nTraining on full dataset...")

# Define columns that shouldn't be used as features in the base model
cols_to_drop = ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'log1p_sales']

# Initialize the custom multi-piecewise regressor
base_model = MultiPiecewiseRegressor(
    my_estimators, 
    store_family_router, 
    verbose=0, 
    n_jobs=-1, # Utilize all available CPU cores
    drop_cols=cols_to_drop
)

# Fit the base model on all available training data
base_model.fit(train, train['log1p_sales'])

# =========================================================
# 6. PREDICTION & SUBMISSION
# =========================================================
print("\nPredicting on test set...")
test_preds = base_model.predict(test)

# Format and save submission
submission_1 = test[['id']].copy()

# Reverse the log1p transformation using expm1 (exp(x) - 1) to get actual sales figures
submission_1['sales'] = np.expm1(test_preds)

submission_1.to_csv('submission_1.csv', index=False)
print("Successfully generated submission_1.csv")